In [ ]:
!pip install -q scikit-learn pandas numpy nltk spacy joblib pypdf
!python -m spacy download en_core_web_sm -q

In [ ]:
import re
import warnings
import joblib
import nltk
import pandas as pd

from nltk.corpus import stopwords as nltk_stopwords
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

for r in ('stopwords', 'wordnet', 'averaged_perceptron_tagger_eng'):
    nltk.download(r, quiet=True)

STOP_WORDS = set(nltk_stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

print('Libraries loaded.')

In [ ]:
def _wordnet_pos(tag):
    return {'J': wordnet.ADJ, 'V': wordnet.VERB, 'N': wordnet.NOUN, 'R': wordnet.ADV}.get(tag[0], wordnet.NOUN)

def preprocess(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    words = text.split()
    pos_tags = nltk.pos_tag(words)
    cleaned = [
        LEMMATIZER.lemmatize(word, _wordnet_pos(tag))
        for word, tag in pos_tags
        if word not in STOP_WORDS and len(word) > 1
    ]
    return ' '.join(cleaned)

print('Preprocessing function ready.')

In [ ]:
df = pd.read_csv('Resume.csv')
print(f'Loaded {len(df)} resumes across {df["Category"].nunique()} categories.')
df.head(2)

In [ ]:
most_freq      = df['Category'].mode()[0]
df['Category'] = df['Category'].fillna(most_freq)

le             = LabelEncoder()
df['Category'] = le.fit_transform(df['Category'])

print('Categories:', list(le.classes_))

In [ ]:
print('Preprocessing resumes — this takes a few minutes...')
df['cleaned_text'] = df['Resume_str'].apply(preprocess)
print('Done.')

In [ ]:
X = df['cleaned_text']
y = df['Category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=2500, ngram_range=(1, 2), sublinear_tf=True)),
    ('clf',   RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)),
])

print('Training...')
pipeline.fit(X_train, y_train)
print('Done.')

In [ ]:
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
joblib.dump(pipeline, 'resume_model.joblib')
joblib.dump(le,       'label_encoder.joblib')
print('Saved resume_model.joblib and label_encoder.joblib')

In [ ]:
from google.colab import files
files.download('resume_model.joblib')
files.download('label_encoder.joblib')
print('Download started. Place both files in your project folder next to app.py.')